In [ ]:
# Importar librerías
import pandas as pd
import openpyxl
import os
import glob

# Datos del gas

In [ ]:
# Función para tratar los datos anuales
def cargar_mibgas_anual(path_archivo, año):
    
    extension = os.path.splitext(path_archivo)[1].lower()
    
    if extension == ".csv":
        df = pd.read_csv(path_archivo, sep=";", skiprows=1)
        df = df[df["Product"] == "GDAES_D+1"].copy()
        df = df.dropna(subset=["MIBGAS Daily Price [EUR/MWh]"])
        df["Last Day Delivery"] = pd.to_datetime(
            df["Last Day Delivery"], format="%d/%m/%Y", errors="coerce"
        )
        df = df[["Last Day Delivery", "MIBGAS Daily Price [EUR/MWh]"]].copy()
        df.columns = ["Fecha", "Precio_EUR_MWh"]

    elif extension in [".xlsx", ".xls"]:
        xl = pd.ExcelFile(path_archivo)
        
        # Buscar la hoja correcta: "Trading Data" o "Trading Data PVB"
        # Excluir hojas de otros mercados (LNG, TVB, AVB)
        hoja = next(
            s for s in xl.sheet_names
            if s.startswith("Trading Data")
            and "LNG" not in s
            and "TVB" not in s
            and "AVB" not in s
        )
        
        df = xl.parse(hoja)
        # Limpiar saltos de línea en nombres de columnas
        df.columns = [str(c).replace("\n", " ").strip() for c in df.columns]
        df = df[df["Product"] == "GDAES_D+1"].copy()
        # Buscar columna de precio dinámicamente por si varía el espaciado
        col_precio = [c for c in df.columns if "Daily Reference Price" in c][0]
        df = df.dropna(subset=[col_precio])
        # La fecha ya viene como datetime, no hace falta parsear formato
        df["Last Day Delivery"] = pd.to_datetime(df["Last Day Delivery"], errors="coerce")
        df = df[["Last Day Delivery", col_precio]].copy()
        df.columns = ["Fecha", "Precio_EUR_MWh"]

    else:
        raise ValueError(f"Formato no soportado: {extension}")
    
    df["Año"] = año
    df = df.sort_values("Fecha").reset_index(drop=True)
    
    return df

In [ ]:
# Cargar los datos anuales
base_path = "/Users/miguel.ros/Desktop/PANELES/Gas y pretróleo/Gas/"

archivos = (
    glob.glob(base_path + "MIBGAS_Data_*.csv") +
    glob.glob(base_path + "MIBGAS_Data_*.xlsx")
)

lista_dfs = []

for archivo in archivos:
    año = int(archivo.split("_")[-1].split(".")[0])
    df = cargar_mibgas_anual(archivo, año)
    lista_dfs.append(df)

gas_total = pd.concat(lista_dfs, ignore_index=True).sort_values("Fecha")

gas_total

In [ ]:
# Ver años
print(gas_total["Año"].unique())

In [ ]:
# Exportar
gas_total.to_csv("/Users/miguel.ros/Desktop/PANELES/Gas y petróleo/Gas/gas_total.csv", index=False)

# Datos del petróleo

In [ ]:
# Cargar los datos de Investing
petroleo_raw1 = pd.read_csv("/Users/miguel.ros/Desktop/PANELES/Gas y petróleo/Petróleo/petroleo_brent1.csv")
petroleo_raw1

petroleo_raw2 = pd.read_csv("/Users/miguel.ros/Desktop/PANELES/Gas y petróleo/Petróleo/petroleo_brent2.csv")
petroleo_raw2

In [ ]:
# Seleccionar y renombrar solo las columnas útiles
petroleo = petroleo_raw1[["Fecha", "Último"]].rename(columns={"Último": "precio"})
petroleo2 = petroleo_raw2[["Fecha", "Último"]].rename(columns={"Último": "precio"})

# Parsear fecha (formato DD.MM.YYYY)
petroleo["fecha"] = pd.to_datetime(petroleo["Fecha"], format="%d.%m.%Y")
petroleo2["fecha"] = pd.to_datetime(petroleo2["Fecha"], format="%d.%m.%Y")

# Convertir precio: quitar comas decimales y pasar a float
petroleo["precio"] = petroleo["precio"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)
petroleo2["precio"] = petroleo2["precio"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)

# Resultado final ordenado por fecha
petroleo = petroleo[["fecha", "precio"]].sort_values("fecha").reset_index(drop=True)
petroleo2 = petroleo2[["fecha", "precio"]].sort_values("fecha").reset_index(drop=True)

data_petroleo = pd.concat([petroleo, petroleo2], ignore_index=True)

data_petroleo = data_petroleo.drop_duplicates()

In [ ]:
data_petroleo

In [ ]:
# Exportar
data_petroleo.to_csv("/Users/miguel.ros/Desktop/PANELES/Gas y petróleo/Petróleo/petroleo.csv", index=False)